# DCM Projects - Private Preview - Demo 2

## Platform Developer - Role Setup
Create a dedicated developer role to manage DCM Projects on a DEV environment.

In [ ]:
%%sql -r role_setup
use role ACCOUNTADMIN;

create role if not exists DCM_ADMIN;
grant role DCM_ADMIN to role ACCOUNTADMIN;
set USER_NAME = (select current_user());
grant role DCM_ADMIN to user identifier($USER_NAME);

In [ ]:
%%sql -r grant_dcm_project
-- to create new dcm project
grant usage on database DCM_DEMO to role DCM_ADMIN;
grant usage on schema DCM_DEMO.PROJECTS to role DCM_ADMIN;
grant create dcm project on schema DCM_DEMO.PROJECTS to role DCM_ADMIN;

In [ ]:
%%sql -r grant_infrastructure
-- to create new account infrastructure through DCM deployments
grant CREATE WAREHOUSE on account to role DCM_ADMIN;
grant CREATE ROLE on account to role DCM_ADMIN;
grant CREATE DATABASE on account to role DCM_ADMIN;
-- create database automatically allows the role to also create schema and all schema-level objects  
grant EXECUTE MANAGED TASK on account to role DCM_ADMIN;
grant EXECUTE TASK on account to role DCM_ADMIN;

In [ ]:
%%sql -r grant_dq
-- if you want to set and test data quality expectations you also need:
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_ADMIN;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_ADMIN;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_ADMIN;
grant execute data metric function on account to role DCM_ADMIN;

In [ ]:
%%sql -r switch_role
-- access to the incoming data share 
grant imported privileges on database TPCDI_SOURCE to role DCM_ADMIN;
 
use role DCM_ADMIN;

## Platform Project Setup

In [ ]:
%%sql -r create_wh
create warehouse if not exists DCM_PLATFORM_WH
with
    warehouse_size = 'XSMALL'
    auto_suspend = 300
    comment = 'For Quickstart Demo of DCM Projects PrPr'
;

In [ ]:
%%sql -r create_db_schema
create database if not exists DCM_DEMO;
create schema if not exists DCM_DEMO.PROJECTS;

In [ ]:
%%sql -r create_dcm_project
create dcm project if not exists DCM_DEMO.PROJECTS.DCM_PLATFORM_DEV
    comment = 'for DCM Demos'
;

In [ ]:
%%sql -r stage_result
-- Copy all CSV files from demo repo to the stage
COPY FILES INTO 
    @DCM_DEMO_2.INGEST.DCM_SAMPLE_DATA
FROM 
    'snow://workspace/USER$.PUBLIC.DCM_DEMO/versions/live/Quickstarts/DCM_Project_Quickstart_2/sample_data'
DETAILED_OUTPUT = TRUE
;

In [ ]:
%%sql -r dataframe_3
execute task DCM_DEMO_2.INGEST.LOAD_NEW_DATA;

## New Workspace UI for DCM Projects

### Plan DCM project
1. In the DCM control panel above the tabs, select right project folder to plan from 
2. Select the configuration profile from the dropdown
3. Click the "Plan" button
4. Connect your definition to an existing DCM Project object (like DCM_PROJECT_DEV in DCM_DEMO/PROJECTS) or create a new project

### Review the plan output
- Try the AI Summary button
- Click 'Deploy' when the Plan output looks right

In [ ]:
%%sql -r use_dcm_admin
use role DCM_ADMIN;

### Test all Data Quality Expectations
Test all Data Quality Expectations set on table objects inside the project.

In [ ]:
%%sql -r platform_test_results
execute dcm project DCM_DEMO.PROJECTS.DCM_PLATFORM_DEV
    TEST ALL

-- optional flow operator to format TEST ALL output and add AI summary
->> with TEST_RESULTS as (select "message" from $1)
    select
        f.value:table_name ::string as TABLE_NAME,
        f.value:arguments ::string as COLUMN_NAME,
        f.value:metric_name ::string as METRIC_NAME,
        f.value:expectation_name ::string as EXPECTATION_NAME,
        f.value:expectation_expression ::string as EXPECTATION_EXPRESSION,
        f.value:value ::number AS VALUE,
        f.value:expectation_violated ::boolean AS EXPECTATION_VIOLATED
    from
        TEST_RESULTS,
        LATERAL FLATTEN(input => PARSE_JSON(TEST_RESULTS."message"):expectations) f    
;

## DT Pipeline Project Setup

In [ ]:
%%sql -r grant_finance_dq
use role ACCOUNTADMIN;

grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role FINANCE_ADMIN;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role FINANCE_ADMIN;
grant database role SNOWFLAKE.DATA_METRIC_USER to role FINANCE_ADMIN;
grant execute data metric function on account to role FINANCE_ADMIN;

In [ ]:
%%sql -r create_pipeline_project
use role FINANCE_ADMIN;

create dcm project if not exists FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV
    comment = 'for DCM Pipeline Demos'
;

### PLAN and DEPLOY
Test the definitions with the DEV configuration against the current state on the account, then deploy.

In [ ]:
%%sql -r refresh_result
-- Refresh all Dynamic Tables defined inside the project to process new sample data
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV 
    REFRESH ALL

    -- optional flow operator to format REFRESH ALL output and add AI summary
->> with REFRESH_RESULT as (select "result" from $1)
    select
        f.value:data_timestamp ::string as TIMESTAMP_IN_MS,
        f.value:dt_name ::string as DT_NAME,
        f.value:statistics ::string as RESULT
    from
        REFRESH_RESULT,
        LATERAL FLATTEN(input => PARSE_JSON(REFRESH_RESULT."result"):refreshed_tables) f
;

In [ ]:
%%sql -r pipeline_test_results
-- TEST all Data Quality Expectations set on table objects inside the project
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV 
    TEST ALL

-- optional flow operator to format TEST ALL output and add AI summary
->> with TEST_RESULTS as (select "message" from $1)
    select
        f.value:table_name ::string as TABLE_NAME,
        f.value:metric_database ::string as METRIC_DATABASE,
        f.value:metric_schema ::string as METRIC_SCHEMA,
        f.value:metric_name ::string as METRIC_NAME,
        f.value:expectation_name ::string as EXPECTATION_NAME,
        f.value:expectation_expression ::string as EXPECTATION_EXPRESSION,
        f.value:value ::number AS VALUE,
        f.value:expectation_violated ::boolean AS EXPECTATION_VIOLATED
    from
        TEST_RESULTS,
        LATERAL FLATTEN(input => PARSE_JSON(TEST_RESULTS."message"):expectations) f
;

### DCM PREVIEW Command
Preview a data sample for a defined table / dynamic table / view to validate transformation logic before deploying changes.

In [ ]:
%%sql -r preview_result
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV
    PREVIEW DCM_DEMO_2_FINANCE_DEV.GOLD.FACT_PROSPECT
    using configuration DEV
    from
        'snow://workspace/USER$.PUBLIC.DCM_DEMO/versions/live/Quickstarts/DCM_Project_Quickstart_2/DCM_Pipeline_Demo'
;

---

# Plan and Deploy PROD Instance

In [ ]:
%%sql -r create_prod_role
-- create service role to run DCM Project deployments on PROD 
use role ACCOUNTADMIN;

create role if not exists DCM_PROD_DEPLOYER;

grant USAGE on database DCM_DEMO to role DCM_PROD_DEPLOYER;
grant USAGE on schema DCM_DEMO.PROJECTS to role DCM_PROD_DEPLOYER;
grant CREATE DCM PROJECT on schema DCM_DEMO.PROJECTS to role DCM_PROD_DEPLOYER;

In [ ]:
%%sql -r grant_prod_infra
-- to create new account infrastructure through DCM deployments
grant CREATE WAREHOUSE on account to role DCM_PROD_DEPLOYER;
grant CREATE ROLE on account to role DCM_PROD_DEPLOYER;
grant CREATE DATABASE on account to role DCM_PROD_DEPLOYER;

In [ ]:
%%sql -r grant_prod_dq
-- if you want to set and test data quality expectations
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_PROD_DEPLOYER;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_PROD_DEPLOYER;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_PROD_DEPLOYER;
grant execute data metric function on account to role DCM_PROD_DEPLOYER;

In [ ]:
%%sql -r assign_prod_role
grant role DCM_PROD_DEPLOYER to user GITHUB_ACTIONS_SERVICE_USER;   -- replace with your CI/CD service-user and/or user name

use role DCM_PROD_DEPLOYER;

In [ ]:
%%sql -r create_prod_project
create or replace dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_PROD
        log_level = INFO
        comment = 'Simulating deployment to PROD with a service-user'
;

### PLAN the new version with the PROD configuration
See the jinja_demo file looping through the list of teams.

In [ ]:
%%sql -r plan_prod
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_PROD 
    PLAN
    using configuration PROD    -- now with PROD config
    from
        'snow://workspace/USER$.PUBLIC.DCM_INTERNAL/versions/live/Quickstarts/DCM_Project_Quickstart_9_35'
        
    -- flatten json output to table
    ->> with JSON_INPUT as (
            select parse_json($1) as OPERATIONS from $1) 
        select
            f.INDEX ::number as INDEX,
            f.value:operationType ::string as OPERATION_TYPE,
            coalesce( f.value:objectDomain, f.value:association) ::string as OBJECT_TYPE,
            coalesce( f.value:objectName, f.value:target:objectName) ::string as OBJECT_NAME,
            coalesce( f.value:details:properties, concat(f.value:subject:objectPrivilege,' on ',f.value:subject:objectName)) ::string as PROPERTIES,
            f.value:details:columns ::string as COLUMNS,
            f.value ::string as VALUE
        from
            JSON_INPUT,
            LATERAL FLATTEN(input => JSON_INPUT.OPERATIONS)f
        order by
            INDEX
;

In [ ]:
%%sql -r deploy_prod
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_PROD 
    DEPLOY
    using configuration PROD
    from
        'snow://workspace/USER$.PUBLIC.DCM_INTERNAL/versions/live/Quickstarts/DCM_Project_Quickstart_9_35'
;

**Note:** You need a dedicated DCM PROJECT object for each environment so that these deployed instances can co-exist! Each DCM Project can only deploy 1 profile at a time.